## 1. Check GPU (Must be L4 or Ampere+)

In [ ]:
import torch
import subprocess

print("="*60)
print("GPU CHECK")
print("="*60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU: {gpu_name}")
    
    # Get compute capability
    major, minor = torch.cuda.get_device_capability(0)
    print(f"   Compute Capability: sm_{major}{minor}")
    
    # Check GPU memory
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"   Total GPU Memory: {total_mem:.1f} GB")
    
    # Check if FlashAttention compatible (Ampere = sm_80+)
    if major >= 8:
        print(f"\n✅ FlashAttention SUPPORTED (Ampere or newer)")
        if total_mem >= 20:
            print(f"✅ Sufficient memory for 3 servers (~6-9GB total)")
        else:
            print(f"⚠️  Limited memory - may only run 2 servers")
    else:
        print(f"\n❌ FlashAttention NOT SUPPORTED!")
        print(f"   T4 (sm_75) and older GPUs cannot use FlashAttention")
        print(f"   Go to Runtime > Change runtime type > L4 GPU")
        raise RuntimeError("Please switch to L4 GPU for FlashAttention support")
else:
    print("❌ No GPU! Go to Runtime > Change runtime type > L4 GPU")
    raise RuntimeError("GPU required")

print("\n" + "="*60)

## 2. Install Dependencies

In [ ]:
# Install all dependencies
import subprocess
import sys

print("📦 Installing system dependencies...")
!apt-get update -qq
!apt-get install -y git wget net-tools > /dev/null 2>&1

print("📦 Installing PyTorch 2.4.0+cu121 (compatible with flash-attn)...")
!pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

print("📦 Installing other dependencies...")
!pip install -q "transformers>=4.45.0" "timm>=1.0.0" "accelerate>=1.0.0" "diffusers>=0.31.0"
!pip install -q websockets pyyaml opencv-python pillow "numpy>=1.26.4,<2.0" fvcore
!pip install -q einops thop cloudpickle easydict hydra-core
!pip install -q huggingface_hub
!pip install -q pyngrok ninja packaging

print("✅ All dependencies installed!")

## 3. Clone Evo-1 Repository

In [ ]:
import os
from pathlib import Path

# Clone Evo-1
if not Path("/content/Evo-1").exists():
    print("📦 Cloning Evo-1 repository...")
    !git clone https://github.com/MINT-SJTU/Evo-1.git /content/Evo-1
    print("✅ Evo-1 cloned")
    
    # Install Evo-1 requirements (skip torch - already installed)
    print("📦 Installing Evo-1 requirements...")
    !pip install -q -r /content/Evo-1/Evo_1/requirements.txt
    print("✅ Evo-1 requirements installed")
else:
    print("✅ Evo-1 already exists")

# Create checkpoint directories
os.makedirs("/content/Evo-1/checkpoints/libero", exist_ok=True)
os.makedirs("/content/Evo-1/checkpoints/metaworld", exist_ok=True)
os.makedirs("/content/Evo-1/checkpoints/robotwin", exist_ok=True)

print("✅ Checkpoint directories created")

## 4. Download All Checkpoints

Downloads checkpoints for all available benchmarks in parallel.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import concurrent.futures
import time

# Checkpoint configurations
CHECKPOINTS = {
    "libero": {
        "repo_id": "MINT-SJTU/Evo1_LIBERO",
        "local_dir": "/content/Evo-1/checkpoints/libero",
        "status": "available"
    },
    "metaworld": {
        "repo_id": "MINT-SJTU/Evo1_MetaWorld",
        "local_dir": "/content/Evo-1/checkpoints/metaworld",
        "status": "available"
    },
    "robotwin": {
        "repo_id": None,  # Not yet released
        "local_dir": "/content/Evo-1/checkpoints/robotwin",
        "status": "coming_soon"  # Check MINT-SJTU GitHub for updates
    }
}

def download_checkpoint(name, config):
    """Download a single checkpoint."""
    if config["status"] != "available":
        return name, "skipped", "Not yet released"
    
    local_dir = config["local_dir"]
    if Path(f"{local_dir}/config.json").exists():
        return name, "exists", "Already downloaded"
    
    try:
        snapshot_download(
            repo_id=config["repo_id"],
            local_dir=local_dir,
            local_dir_use_symlinks=False
        )
        return name, "success", f"Downloaded to {local_dir}"
    except Exception as e:
        return name, "error", str(e)

print("📥 Downloading checkpoints (this may take a few minutes)...\n")

# Download in parallel
start_time = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    futures = {
        executor.submit(download_checkpoint, name, config): name 
        for name, config in CHECKPOINTS.items()
    }
    
    for future in concurrent.futures.as_completed(futures):
        name, status, message = future.result()
        if status == "success":
            print(f"✅ {name.upper()}: {message}")
        elif status == "exists":
            print(f"✅ {name.upper()}: {message}")
        elif status == "skipped":
            print(f"⏳ {name.upper()}: {message}")
        else:
            print(f"❌ {name.upper()}: {message}")

elapsed = time.time() - start_time
print(f"\n⏱️  Completed in {elapsed:.1f} seconds")

## 5. Install flash-attn

⚠️ **After this cell completes, you MUST restart the runtime!**

In [ ]:
# Install flash-attn (critical for performance)
print("🔧 Installing flash-attn...")
print("   This takes 1-2 minutes...\n")

# Ensure PyTorch 2.4.0+cu121 is still installed
!pip install -q torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

# Install flash-attn
!pip install flash-attn --no-build-isolation 2>&1 | tail -10

print("\n" + "="*60)
print("⚠️  IMPORTANT: RESTART RUNTIME NOW!")
print("="*60)
print("\nGo to: Runtime > Restart runtime")
print("Then run cells starting from cell 7 (Verify Dependencies)")
print("="*60)

## 6. (Skip) Placeholder for additional setup

In [ ]:
# Placeholder - skip this cell
print("⏭️  Skipping - this is a placeholder cell")

## 7. Verify Dependencies (Run AFTER Runtime Restart)

In [ ]:
# Verify torch and flash-attn after restart
import torch

print("="*60)
print("DEPENDENCY VERIFICATION")
print("="*60)

print(f"\n✅ PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

# Verify flash-attn
try:
    import flash_attn
    print(f"✅ flash-attn: {flash_attn.__version__}")
    flash_attn_available = True
except ImportError:
    print("❌ flash-attn not available")
    print("   Re-run cell 5 (flash-attn installation) and restart runtime again")
    flash_attn_available = False

# Verify GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU: {gpu_name} ({total_mem:.1f} GB)")
else:
    print("❌ CUDA not available!")

# Verify checkpoints exist
from pathlib import Path
print("\n📁 Checkpoints:")
for name in ["libero", "metaworld", "robotwin"]:
    ckpt_path = f"/content/Evo-1/checkpoints/{name}/config.json"
    if Path(ckpt_path).exists():
        print(f"   ✅ {name}: Found")
    else:
        print(f"   ⏳ {name}: Not available")

print("\n" + "="*60)
if flash_attn_available:
    print("✅ Ready to start servers!")
else:
    print("⚠️  flash-attn missing - servers will be slower")
print("="*60)

## 8. Configure ngrok

In [ ]:
# Set your ngrok authtoken here
# Get free token from: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- REPLACE THIS

from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN == "YOUR_NGROK_TOKEN_HERE":
    print("❌ Please set your ngrok auth token!")
    print("   Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok configured")

## 9. Create Multi-Server Script

This creates a modified server script that can run on different ports with different checkpoints.

In [ ]:
import json
import os

# Server configuration
SERVERS = {
    "libero": {
        "port": 9001,
        "checkpoint": "/content/Evo-1/checkpoints/libero",
        "enabled": True
    },
    "metaworld": {
        "port": 9002,
        "checkpoint": "/content/Evo-1/checkpoints/metaworld",
        "enabled": True
    },
    "robotwin": {
        "port": 9003,
        "checkpoint": "/content/Evo-1/checkpoints/robotwin",
        "enabled": False  # Not yet released
    }
}

# Read the original server script
with open("/content/Evo-1/Evo_1/scripts/Evo1_server.py", 'r') as f:
    original_server = f.read()

# Create a parameterized version
multi_server_script = '''#!/usr/bin/env python3
"""
Evo-1 Multi-Server Script
Usage: python Evo1_multi_server.py --port PORT --checkpoint CHECKPOINT_DIR --name SERVER_NAME
"""
import argparse
import sys
import os

# Parse arguments BEFORE importing other modules
parser = argparse.ArgumentParser(description="Evo-1 Server")
parser.add_argument("--port", type=int, required=True, help="Server port")
parser.add_argument("--checkpoint", type=str, required=True, help="Checkpoint directory")
parser.add_argument("--name", type=str, default="evo1", help="Server name")
args = parser.parse_args()

# Now import and modify the original server
'''

# Modify original server to use args
modified_server = original_server

# Replace hardcoded checkpoint path
modified_server = modified_server.replace(
    'ckpt_dir = "Your/Path/To/Checkpoint"',
    'ckpt_dir = args.checkpoint'
)
modified_server = modified_server.replace(
    'ckpt_dir = "/content/Evo-1/checkpoints/libero"',
    'ckpt_dir = args.checkpoint'
)

# Replace hardcoded port
modified_server = modified_server.replace(
    'port = 9000',
    'port = args.port'
)
modified_server = modified_server.replace(
    'port = 9001',
    'port = args.port'
)
modified_server = modified_server.replace(
    'port=9000',
    'port=args.port'
)
modified_server = modified_server.replace(
    'port=9001',
    'port=args.port'
)

# Add server name to startup message
modified_server = modified_server.replace(
    'print(f"WebSocket server',
    'print(f"[{args.name}] WebSocket server'
)

# Combine the arg parser with the modified script
final_script = multi_server_script + modified_server

# Write the multi-server script
multi_server_path = "/content/Evo-1/Evo_1/scripts/Evo1_multi_server.py"
with open(multi_server_path, 'w') as f:
    f.write(final_script)

print("✅ Multi-server script created")

# Update config files for each checkpoint
for name, config in SERVERS.items():
    config_path = f"{config['checkpoint']}/config.json"
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            ckpt_config = json.load(f)
        ckpt_config['device'] = 'cuda'
        with open(config_path, 'w') as f:
            json.dump(ckpt_config, f, indent=2)
        print(f"✅ {name.upper()} config updated")
    else:
        print(f"⏳ {name.upper()} checkpoint not available")

print("\n✅ Server configuration complete")

## 10. Kill Any Existing Servers

Run this before starting new servers.

In [ ]:
# Kill any existing servers and ngrok tunnels
import subprocess
import time

print("🔄 Cleaning up existing processes...\n")

# Kill ngrok tunnels
try:
    from pyngrok import ngrok
    tunnels = ngrok.get_tunnels()
    for tunnel in tunnels:
        ngrok.disconnect(tunnel.public_url)
    ngrok.kill()
    print("✅ ngrok tunnels killed")
except Exception as e:
    print(f"⚠️  ngrok cleanup: {e}")

# Kill existing Evo-1 servers
!pkill -9 -f "Evo1_server" 2>/dev/null || true
!pkill -9 -f "Evo1_multi_server" 2>/dev/null || true
print("✅ Evo-1 processes killed")

# Kill anything on our ports
for port in [9001, 9002, 9003]:
    !kill -9 $(lsof -t -i:{port}) 2>/dev/null || true
print("✅ Ports 9001-9003 freed")

time.sleep(2)

# Verify ports are free
print("\n📋 Port status:")
for port in [9001, 9002, 9003]:
    result = !netstat -tuln 2>/dev/null | grep :{port}
    if result:
        print(f"   ⚠️  Port {port}: In use")
    else:
        print(f"   ✅ Port {port}: Free")

print("\n✅ Cleanup complete - ready to start servers")

## 11. Start All Servers with ngrok Tunnels

This starts all enabled servers in parallel with separate ngrok tunnels.

In [ ]:
import subprocess
import time
import os
from pyngrok import ngrok
from pathlib import Path

# Server configuration
SERVERS = {
    "libero": {
        "port": 9001,
        "checkpoint": "/content/Evo-1/checkpoints/libero",
        "enabled": True
    },
    "metaworld": {
        "port": 9002,
        "checkpoint": "/content/Evo-1/checkpoints/metaworld",
        "enabled": True
    },
    "robotwin": {
        "port": 9003,
        "checkpoint": "/content/Evo-1/checkpoints/robotwin",
        "enabled": False  # Set to True when RoboTwin checkpoint is released
    }
}

# Ensure dependencies are installed
print("📦 Installing server dependencies...")
!pip install -q websockets opencv-python pillow transformers timm accelerate diffusers fvcore einops

# Verify flash-attn
import torch
print(f"\n📌 PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
try:
    import flash_attn
    print(f"✅ flash-attn: {flash_attn.__version__}")
except ImportError:
    print("⚠️  flash-attn not available - servers will be slower")

# Track server processes and tunnels
server_processes = {}
server_tunnels = {}

print("\n" + "="*60)
print("STARTING SERVERS")
print("="*60)

# Start each enabled server
for name, config in SERVERS.items():
    if not config["enabled"]:
        print(f"\n⏭️  {name.upper()}: Skipped (not enabled)")
        continue
    
    # Check if checkpoint exists
    if not Path(f"{config['checkpoint']}/config.json").exists():
        print(f"\n❌ {name.upper()}: Checkpoint not found at {config['checkpoint']}")
        continue
    
    port = config["port"]
    checkpoint = config["checkpoint"]
    
    print(f"\n🚀 Starting {name.upper()} server on port {port}...")
    
    # Create log file
    log_file = f"/content/server_{name}.log"
    
    # Start server process
    with open(log_file, 'w') as log:
        process = subprocess.Popen(
            [
                "python", "Evo_1/scripts/Evo1_multi_server.py",
                "--port", str(port),
                "--checkpoint", checkpoint,
                "--name", name
            ],
            cwd="/content/Evo-1",
            stdout=log,
            stderr=subprocess.STDOUT
        )
    
    server_processes[name] = {
        "process": process,
        "port": port,
        "log": log_file
    }
    print(f"   ✅ Process started (PID: {process.pid})")
    print(f"   📝 Log: {log_file}")

# Wait for servers to start
print("\n⏳ Waiting for servers to load models (2-3 minutes each)...")

all_ready = False
for i in range(36):  # Check every 5 seconds for 3 minutes
    time.sleep(5)
    
    ready_count = 0
    for name, info in server_processes.items():
        # Check if process crashed
        if info["process"].poll() is not None:
            print(f"\n❌ {name.upper()} crashed!")
            print(f"   Last 20 lines of log:")
            !tail -20 {info['log']}
            continue
        
        # Check if port is listening
        result = subprocess.run(
            f"netstat -tuln 2>/dev/null | grep :{info['port']}",
            shell=True, capture_output=True, text=True
        )
        if result.stdout.strip():
            ready_count += 1
    
    if ready_count == len(server_processes):
        all_ready = True
        break
    
    if i % 3 == 0:
        print(f"   ⏳ {(i+1)*5}s elapsed... ({ready_count}/{len(server_processes)} ready)")

if not all_ready:
    print("\n⚠️  Some servers may not be ready yet, but continuing with ngrok setup...")

# Create ngrok tunnels for each server
print("\n🌐 Creating ngrok tunnels...")

for name, info in server_processes.items():
    try:
        tunnel = ngrok.connect(info["port"], bind_tls=True)
        public_url = tunnel.public_url.replace('https://', 'wss://')
        server_tunnels[name] = public_url
        print(f"   ✅ {name.upper()}: {public_url}")
    except Exception as e:
        print(f"   ❌ {name.upper()}: Failed to create tunnel - {e}")

# Print summary
print("\n" + "="*60)
print("✅ ALL SERVERS READY!")
print("="*60)
print("\n📋 SERVER URLS FOR CLIENTS:")
print("-" * 60)

for name, url in server_tunnels.items():
    print(f"\n🔹 {name.upper()}:")
    print(f"   SERVER_URL = '{url}'")

print("\n" + "-" * 60)
print("\n📝 CLIENT COMMANDS (run on your local machine):")
print("-" * 60)

if "libero" in server_tunnels:
    print(f"\n# LIBERO Evaluation:")
    print(f"# Update SERVER_URL in libero_client_4tasks.py to:")
    print(f"# SERVER_URL = '{server_tunnels['libero']}'")
    print(f"conda activate libero")
    print(f"python libero_client_4tasks.py")

if "metaworld" in server_tunnels:
    print(f"\n# MetaWorld Evaluation:")
    print(f"# Update SERVER_URL in mt50_evo1_client_prompt.py to:")
    print(f"# SERVER_URL = '{server_tunnels['metaworld']}'")
    print(f"conda activate metaworld")
    print(f"python mt50_evo1_client_prompt.py")

if "robotwin" in server_tunnels:
    print(f"\n# RoboTwin Evaluation:")
    print(f"# SERVER_URL = '{server_tunnels['robotwin']}'")
    print(f"# (Client script TBD)")

print("\n" + "="*60)
print("⚠️  Keep this cell running during evaluation!")
print("="*60)

## 12. Monitor Server Logs

In [ ]:
# View logs for all servers
import os

for name in ["libero", "metaworld", "robotwin"]:
    log_file = f"/content/server_{name}.log"
    if os.path.exists(log_file):
        print(f"\n{'='*60}")
        print(f"📋 {name.upper()} Server Log (last 15 lines):")
        print(f"{'='*60}")
        !tail -15 {log_file}
    else:
        print(f"\n⏭️  {name.upper()}: No log file (server not started)")

## 13. Check GPU Memory Usage

In [ ]:
# Check GPU memory usage
!nvidia-smi

# Python-based check
import torch
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / (1024**3)
    reserved = torch.cuda.memory_reserved(0) / (1024**3)
    total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    
    print(f"\n📊 GPU Memory Summary:")
    print(f"   Allocated: {allocated:.2f} GB")
    print(f"   Reserved:  {reserved:.2f} GB")
    print(f"   Total:     {total:.1f} GB")
    print(f"   Free:      {total - reserved:.2f} GB")

## 14. Stop All Servers (When Done)

In [ ]:
# Stop all servers and ngrok tunnels
import subprocess

print("🛑 Stopping all servers...\n")

# Stop server processes
try:
    for name, info in server_processes.items():
        try:
            info["process"].terminate()
            info["process"].wait(timeout=5)
            print(f"✅ {name.upper()} server stopped")
        except Exception as e:
            print(f"⚠️  {name.upper()}: {e}")
except NameError:
    print("⚠️  No server_processes variable - killing by name")

# Force kill any remaining processes
!pkill -9 -f "Evo1_server" 2>/dev/null || true
!pkill -9 -f "Evo1_multi_server" 2>/dev/null || true

# Kill processes on ports
for port in [9001, 9002, 9003]:
    !kill -9 $(lsof -t -i:{port}) 2>/dev/null || true

print("✅ All Evo-1 processes killed")

# Stop ngrok
try:
    from pyngrok import ngrok
    ngrok.kill()
    print("✅ ngrok stopped")
except Exception as e:
    print(f"⚠️  ngrok: {e}")

print("\n✅ All servers stopped")

---

## Appendix: Local Client Setup

### LIBERO Client (Python 3.8.13)

```bash
# Create environment
conda create -n libero python=3.8.13 -y
conda activate libero

# Clone and install LIBERO
git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
cd LIBERO
pip install -r requirements.txt
pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
pip install -e .
pip install websockets huggingface_hub

# Run client (update SERVER_URL first!)
python libero_client_4tasks.py
```

### MetaWorld Client (Python 3.10)

```bash
# Create environment
conda create -n metaworld python=3.10 -y
conda activate metaworld

# Install dependencies
pip install mujoco metaworld
pip install websockets opencv-python packaging huggingface_hub

# Run client (update SERVER_URL first!)
python mt50_evo1_client_prompt.py
```

### RoboTwin Client (Coming Soon)

Check the [Evo-1 GitHub](https://github.com/MINT-SJTU/Evo-1) for updates on RoboTwin evaluation.